# NearMe Clemency Classifier

Unsupervised clustering (K-Means & DBSCAN) to classify environmental and hazard data entries by **clemency** — the mildness or severity of conditions. Combines all 7 data sources (EPA air quality, NOAA weather, USGS water, NCDOT crashes, NCGS landslides, NCEM flood zones, IRWIN wildland fires) into a unified severity-scored dataset and assigns clemency labels from **Benign** to **Extreme**.

In [0]:
# Feature Engineering — Compute severity scores (0=benign, 1=extreme) per source
# Clemency = 1 - severity_score (high clemency = mild, low = severe)

from pyspark.sql.functions import col, lit, when, abs as Fabs, coalesce

# 1. EPA Air Quality — AQI / 500 (EPA scale: 0-50 Good, 51-100 Moderate, ... 301-500 Hazardous)
epa = (spark.table("workspace.default.gold_air_quality_latest")
    .filter(col("us_aqi").isNotNull())
    .select(
        lit("EPA Air Quality").alias("source_type"),
        col("latitude"), col("longitude"),
        (col("us_aqi") / 500.0).alias("severity_score"),
        col("us_aqi").alias("metric_value"),
        col("aqi_category").alias("detail"),
    ))

# 2. NOAA Weather — discomfort index from temp deviation, wind, humidity
# Join with GIS table for station coordinates
noaa_coords = (spark.table("workspace.default.gis_environmental_monitoring")
    .filter(col("source_type") == "NOAA Weather")
    .select(col("metric_label").alias("station_name"),
            col("measurement_time").alias("observation_time"),
            col("latitude"), col("longitude"))
    .dropDuplicates(["station_name"]))

noaa = (spark.table("workspace.default.silver_noaa_weather_observations")
    .join(noaa_coords, on="station_name", how="left")
    .filter(col("latitude").isNotNull())
    .select(
        lit("NOAA Weather").alias("source_type"),
        col("latitude"), col("longitude"),
        ((Fabs(coalesce(col("temperature_f"), lit(70.0)) - lit(70.0)) / lit(60.0)
          + coalesce(col("wind_speed_mph"), lit(0.0)) / lit(50.0)
          + Fabs(coalesce(col("humidity"), lit(50.0)) - lit(50.0)) / lit(50.0)) / lit(3.0)).alias("severity_score"),
        col("temperature_f").alias("metric_value"),
        col("station_name").alias("detail"),
    ))

# 3. USGS Water — normalize per variable type (discharge: /10000, gage height: /20)
usgs = (spark.table("workspace.default.gold_water_conditions")
    .filter(col("value").isNotNull() & (col("value") > -999999))
    .select(
        lit("USGS Water").alias("source_type"),
        col("latitude"), col("longitude"),
        when(col("variable_name").contains("Streamflow"), col("value") / 10000.0)
        .otherwise(col("value") / 20.0).alias("severity_score"),
        col("value").alias("metric_value"),
        col("variable_name").alias("detail"),
    ))

# 4. NCDOT Crashes — base severity 0.3, +0.3 if lanes affected, +0.4 if full closure
ncdot = (spark.table("workspace.default.silver_ncdot_crash_data")
    .select(
        lit("NCDOT Crash").alias("source_type"),
        col("latitude"), col("longitude"),
        (lit(0.3)
         + when(col("lanes_affected").isNotNull() & (col("lanes_affected") != ""), lit(0.3)).otherwise(lit(0.0))
         + when(col("is_full_closure") == True, lit(0.4)).otherwise(lit(0.0))).alias("severity_score"),
        lit(None).cast("double").alias("metric_value"),
        col("condition").alias("detail"),
    ))

# 5. NCGS Landslides — impact factor + damage factor + fatalities
impact_map = when(col("impact") == "Dam", 1.0).when(col("impact").isin("Building", "Dwelling"), 0.8).when(col("impact") == "Utilities", 0.7).when(col("impact") == "Road", 0.5).otherwise(0.3)
damage_map = when(col("damage") == "Destroyed", 1.0).when(col("damage") == "Completely collapsed", 0.9).when(col("damage").contains("Displaced"), 0.85).when(col("damage").contains("Partially destroyed"), 0.8).when(col("damage") == "Partially collapsed", 0.6).when(col("damage").contains("Abutted"), 0.5).when(col("damage") == "Covered by debris", 0.4).when(col("damage") == "Undetermined", 0.3).otherwise(0.2)

landslides = (spark.table("workspace.default.silver_ncgs_landslide_data")
    .select(
        lit("NCGS Landslide").alias("source_type"),
        col("latitude"), col("longitude"),
        (impact_map * 0.4 + damage_map * 0.4 + coalesce(col("fatalities"), lit(0)) / 5.0 * 0.2).alias("severity_score"),
        col("fatalities").cast("double").alias("metric_value"),
        col("impact").alias("detail"),
    ))

# 6. NCEM Flood Zones — zone type factor + depth
zone_map = when(col("zone_type") == "VE", 0.9).when(col("zone_type") == "AE", 0.6).when(col("zone_type") == "A", 0.4).otherwise(0.3)
floods = (spark.table("workspace.default.silver_ncem_flood_zones")
    .select(
        lit("NCEM Flood Zone").alias("source_type"),
        col("latitude"), col("longitude"),
        (zone_map * 0.6 + coalesce(col("depth"), lit(0.0)) / 10.0 * 0.4).alias("severity_score"),
        col("depth").alias("metric_value"),
        col("zone_type").alias("detail"),
    ))

# 7. IRWIN Fires — size + containment + complexity
complexity_map = when(col("complexity_level") == "Type 1 Incident", 1.0).when(col("complexity_level") == "Type 2 Incident", 0.8).when(col("complexity_level") == "Type 3 Incident", 0.6).when(col("complexity_level") == "Type 4 Incident", 0.4).when(col("complexity_level") == "Type 5 Incident", 0.2).otherwise(0.1)

fires = (spark.table("workspace.default.silver_irwin_fire_incidents")
    .select(
        lit("IRWIN Fire").alias("source_type"),
        col("latitude"), col("longitude"),
        (coalesce(col("incident_size"), lit(0.0)) / 2000.0 * 0.4
         + (lit(1.0) - coalesce(col("percent_contained"), lit(100.0)) / 100.0) * 0.3
         + complexity_map * 0.3).alias("severity_score"),
        col("incident_size").alias("metric_value"),
        col("incident_name").alias("detail"),
    ))

# Combine all sources
unified = (epa.unionByName(noaa).unionByName(usgs).unionByName(ncdot)
          .unionByName(landslides).unionByName(floods).unionByName(fires))

# Clip severity to [0, 1] and compute clemency = 1 - severity
unified = unified.withColumn("severity_score", when(col("severity_score") > 1.0, lit(1.0)).otherwise(when(col("severity_score") < 0.0, lit(0.0)).otherwise(col("severity_score"))))
unified = unified.withColumn("clemency_score", lit(1.0) - col("severity_score"))

# Save unified scored data
unified.write.mode("overwrite").saveAsTable("workspace.default.clemency_scored_data")

from pyspark.sql.functions import count as Fcount, avg as Favg

total = unified.count()
print(f"Unified scored dataset: {total} entries")
print("\nSeverity by source:")
display(unified.groupBy("source_type").agg(
    Fcount("*").alias("count"),
    Favg("severity_score").alias("avg_severity"),
    Favg("clemency_score").alias("avg_clemency"),
).orderBy(col("count").desc()))


In [0]:
# K-Means Clustering — classify entries into clemency levels
# Features: severity_score, latitude, longitude (standardized)
# k=5 clusters: Benign, Mild, Moderate, Severe, Extreme

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# Load scored data into pandas
pdf = spark.table("workspace.default.clemency_scored_data").toPandas()
print(f"Loaded {len(pdf)} entries for clustering")

# Features for clustering
feature_cols = ["severity_score", "latitude", "longitude"]
X = pdf[feature_cols].values

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit K-Means with k=5
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
pdf["kmeans_cluster"] = kmeans_labels

# Map clusters to clemency levels based on mean severity
cluster_severity = pdf.groupby("kmeans_cluster")["severity_score"].mean().sort_values()
clemency_labels = ["Benign", "Mild", "Moderate", "Severe", "Extreme"]
cluster_to_label = {cluster: label for cluster, label in zip(cluster_severity.index, clemency_labels)}
pdf["kmeans_clemency"] = pdf["kmeans_cluster"].map(cluster_to_label)

print("\nK-Means Cluster Summary (sorted by severity):")
for cluster, label in zip(cluster_severity.index, clemency_labels):
    subset = pdf[pdf["kmeans_cluster"] == cluster]
    print(f"  {label} (cluster {cluster}): {len(subset)} entries, "
          f"avg severity={subset['severity_score'].mean():.3f}, "
          f"avg clemency={subset['clemency_score'].mean():.3f}")
    print(f"    Sources: {dict(subset['source_type'].value_counts())}")

print("\nClemency label distribution:")
display(pd.DataFrame(pdf["kmeans_clemency"].value_counts().reset_index()
    .rename(columns={"index": "clemency", "kmeans_clemency": "count"})))


In [0]:
# DBSCAN Clustering — discover natural density-based clusters
# DBSCAN doesn't require predefining k; finds noise points automatically
# eps=0.5, min_samples=10 on standardized features

from sklearn.cluster import DBSCAN

# Fit DBSCAN on the same standardized features
dbscan = DBSCAN(eps=0.5, min_samples=10, metric="euclidean")
dbscan_labels = dbscan.fit_predict(X_scaled)
pdf["dbscan_cluster"] = dbscan_labels

n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
noise_points = (dbscan_labels == -1).sum()
print(f"DBSCAN found {n_clusters_dbscan} clusters + {noise_points} noise points")

# Assign clemency labels to DBSCAN clusters by mean severity
if n_clusters_dbscan > 0:
    cluster_severity_db = pdf[pdf["dbscan_cluster"] >= 0].groupby("dbscan_cluster")["severity_score"].mean().sort_values()
    # Map to clemency labels (use same 5 levels or adapt)
    if n_clusters_dbscan <= 5:
        db_labels = clemency_labels[:n_clusters_dbscan]
    else:
        # Too many clusters — group by severity quintiles
        db_labels = [f"C{i+1}" for i in range(n_clusters_dbscan)]
    cluster_to_label_db = {c: l for c, l in zip(cluster_severity_db.index, db_labels)}
    pdf["dbscan_clemency"] = pdf["dbscan_cluster"].map(lambda c: cluster_to_label_db.get(c, "Noise"))
else:
    pdf["dbscan_clemency"] = "Noise"

print("\nDBSCAN Cluster Summary (sorted by severity):")
for cluster in cluster_severity_db.index:
    subset = pdf[pdf["dbscan_cluster"] == cluster]
    label = cluster_to_label_db[cluster]
    print(f"  {label} (cluster {cluster}): {len(subset)} entries, "
          f"avg severity={subset['severity_score'].mean():.3f}")
    print(f"    Sources: {dict(subset['source_type'].value_counts())}")

noise = pdf[pdf["dbscan_cluster"] == -1]
if len(noise) > 0:
    print(f"\n  Noise points: {len(noise)} entries")
    print(f"    Sources: {dict(noise['source_type'].value_counts())}")


In [0]:
# Visualization — Clemency clusters on NC map
# K-Means clusters color-coded by clemency level

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Color map for clemency levels
clemency_colors = {
    "Benign": "#2ecc71", "Mild": "#f1c40f", "Moderate": "#e67e22",
    "Severe": "#e74c3c", "Extreme": "#8e44ad",
    "Noise": "#bdc3c7", "C1": "#f1c40f", "C2": "#e67e22", "C3": "#e74c3c",
    "C4": "#8e44ad", "C5": "#2c3e50",
}

# Plot 1: K-Means clusters
colors_km = pdf["kmeans_clemency"].map(clemency_colors)
axes[0].scatter(pdf["longitude"], pdf["latitude"], c=colors_km, s=10, alpha=0.6, edgecolors="none")
axes[0].set_title("K-Means Clemency Clusters", fontsize=14)
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
legend_handles = [mpatches.Patch(color=clemency_colors[l], label=l) for l in sorted(pdf["kmeans_clemency"].unique())]
axes[0].legend(handles=legend_handles, loc="upper left", fontsize=8)

# Plot 2: DBSCAN clusters
db_colors = pdf["dbscan_clemency"].map(clemency_colors).fillna("#bdc3c7")
axes[1].scatter(pdf["longitude"], pdf["latitude"], c=db_colors, s=10, alpha=0.6, edgecolors="none")
axes[1].set_title("DBSCAN Clemency Clusters", fontsize=14)
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
db_legend = [mpatches.Patch(color=clemency_colors.get(l, "#bdc3c7"), label=l) for l in sorted(pdf["dbscan_clemency"].unique())]
axes[1].legend(handles=db_legend, loc="upper left", fontsize=8)

plt.suptitle("North Carolina Environmental Clemency Classification", fontsize=16, y=1.02)
plt.tight_layout()
display(fig)
plt.close()

# Severity distribution by clemency level
fig2, ax = plt.subplots(figsize=(10, 5))
levels_order = ["Benign", "Mild", "Moderate", "Severe", "Extreme"]
levels_present = [l for l in levels_order if l in pdf["kmeans_clemency"].values]
for i, level in enumerate(levels_present):
    subset = pdf[pdf["kmeans_clemency"] == level]
    ax.bar(i, len(subset), color=clemency_colors[level], label=level)
    ax.text(i, len(subset) + 50, str(len(subset)), ha="center", fontsize=10)
ax.set_xticks(range(len(levels_present)))
ax.set_xticklabels(levels_present)
ax.set_title("Entry Count by K-Means Clemency Level", fontsize=14)
ax.set_ylabel("Number of Entries")
plt.tight_layout()
display(fig2)
plt.close()


In [0]:
# Save labeled data to Delta table
# Creates workspace.default.clemency_labeled with K-Means and DBSCAN labels

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

result_pdf = pdf[["source_type", "latitude", "longitude", "severity_score", "clemency_score",
                  "metric_value", "detail", "kmeans_cluster", "kmeans_clemency",
                  "dbscan_cluster", "dbscan_clemency"]].copy()

result_schema = StructType([
    StructField("source_type", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("severity_score", DoubleType()),
    StructField("clemency_score", DoubleType()),
    StructField("metric_value", DoubleType()),
    StructField("detail", StringType()),
    StructField("kmeans_cluster", IntegerType()),
    StructField("kmeans_clemency", StringType()),
    StructField("dbscan_cluster", IntegerType()),
    StructField("dbscan_clemency", StringType()),
])

result_df = spark.createDataFrame(result_pdf, schema=result_schema)
result_df.write.mode("overwrite").saveAsTable("workspace.default.clemency_labeled")

cnt = spark.sql("SELECT COUNT(*) as cnt FROM workspace.default.clemency_labeled").collect()[0]["cnt"]
print(f"Saved {cnt} labeled entries to workspace.default.clemency_labeled")

print("\nFinal K-Means clemency distribution:")
display(spark.sql("""
    SELECT kmeans_clemency, source_type, COUNT(*) as cnt,
           ROUND(AVG(severity_score), 3) as avg_severity,
           ROUND(AVG(clemency_score), 3) as avg_clemency
    FROM workspace.default.clemency_labeled
    GROUP BY kmeans_clemency, source_type
    ORDER BY kmeans_clemency, cnt DESC
"""))

print("\nSample labeled entries:")
display(spark.sql("""
    SELECT source_type, detail, kmeans_clemency, dbscan_clemency,
           ROUND(severity_score, 3) as severity, ROUND(clemency_score, 3) as clemency,
           latitude, longitude
    FROM workspace.default.clemency_labeled
    ORDER BY severity_score DESC
    LIMIT 20
"""))
